# Full-res CACHE builder + uploader (CPU sessions — parallel workers)

**PARALLEL PLAN: one session per scenario.** Run three copies of this notebook
(yours + teammates', or three of your own — Kaggle allows 5 concurrent CPU sessions),
each running ITS scenario cell below:
- worker ssp126 also builds `X_test.nc` + the hist window (single writer for the shared extras)
- workers ssp370 / ssp585 build only their scenario

ONE dataset per scenario: `climx-cache-ssp126`, `climx-cache-ssp370`, `climx-cache-ssp585`
Each worker has its own manifest (`data/full_cache_manifest_<scenario>.json`) — commit them
to the repo at the end (or send them to None).

**Setup per copy:** Data → Add Input → `climx-full-resolution-data` (fallback only; the
script downloads from HF) → **Accelerator: NONE (CPU)**. No GPU quota is used.

**Runtime per worker ≈ 3–4 h**, resumable via the manifest.

## 1. Clone the repo + install deps (~2 min, skips on reruns)

In [ ]:
import os, subprocess, sys, pathlib
repo = "/kaggle/working/climx"
if os.path.isdir(repo):
    r = subprocess.run(["git", "-C", repo, "pull", "--ff-only"], capture_output=True, text=True)
    print(r.stdout or r.stderr)
else:
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/nonewanit/climx-emulator.git", repo], check=True)
os.chdir(repo)
sys.path.insert(0, str(pathlib.Path(repo) / "src"))
subprocess.run(["find", repo, "-name", "__pycache__", "-type", "d",
                "-exec", "rm", "-rf", "{}", "+"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-kaggle.txt"], check=True)
r = subprocess.run(["git", "-C", repo, "log", "-1", "--oneline"], capture_output=True, text=True)
print("HEAD:", r.stdout.strip(), "— MUST be the commit None told you")
print("setup done")

## 2. Build + upload the cache

## 2. Run YOUR scenario (one session per scenario, in parallel)

In [ ]:
# WORKER 1 — ssp126 + X_test + hist window:
%run scripts/full_cache.py --scenarios ssp126

In [ ]:
# WORKER 2 — ssp370 only:
# %run scripts/full_cache.py --scenarios ssp370

In [ ]:
# WORKER 3 — ssp585 only:
# %run scripts/full_cache.py --scenarios ssp585

## 3. After CACHE_UPLOAD_OK on all workers

Commit the per-scenario manifests to the repo (`data/full_cache_manifest_*.json`) and
share the `climx-cache-*` datasets with the team (dataset pages → Share). The training
notebook attaches them all and copies them to /tmp.

## 3. After CACHE_UPLOAD_OK

The log prints the dataset slugs (`nonewanitchollakit/climx-cache`, …) and the
manifest JSON. Commit the manifest to the repo and share the datasets with the team
(dataset page → Share → add collaborators) so training sessions can attach them.